<a href="https://colab.research.google.com/github/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/blob/main/Notebooks/binary_decision_tree_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**BINARY DECISION TREE ALGORITHM FROM SCRATCH**

**IMPORT LIBRARIES**

In [1]:
import numpy as np
import pandas as pd
from google.colab import sheets


**DEFINE CLASS**

In [111]:
import numpy as np
import pandas as pd


class BinaryDecisionTree:

    def __init__(self):
        """
        Initializes the BinaryDecisionTree instance.

        Attributes:
            tree (list): Instance-level list that stores the tree structure as a
                         sequence of node dictionaries built during fitting. Each
                         element is a dict produced by get_node().
        """
        self.tree = []

    def get_filter_labels(self, attribute, labels, valid, rule, condition):
        """
        Filters labels based on a threshold rule applied to an attribute column,
        using a two-flag logic (valid, condition) to select the comparison direction.

        The effective comparison is determined by the combination of `valid` and
        `condition` according to the following truth table:

            valid=1, condition=0  →  attribute < rule   (left / "less-than" branch)
            valid=1, condition=1  →  attribute > rule   (right / "greater-than" branch)
            valid=0, condition=0  →  attribute > rule   (complement of condition=0)
            valid=0, condition=1  →  attribute < rule   (complement of condition=1)

        This mirrors the branching logic used in get_filter_attributes so that both
        methods produce consistent subsets from the same (valid, rule, condition) triple.

        Parameters:
            attribute (numpy.ndarray): 1-D array of attribute values, already sorted
                                       in the same order as `labels`.
            labels (numpy.ndarray): 1-D array of class labels corresponding to each
                                    row of `attribute`.
            valid (int): Branch selector. 1 selects the "primary" direction defined
                         by `condition`; 0 selects the complementary direction.
            rule (float): Threshold value used in the comparison.
            condition (int): Comparison direction flag. 0 means the primary comparison
                             is `<`; 1 means the primary comparison is `>`.

        Returns:
            numpy.ndarray: Subset of `labels` whose corresponding attribute values
                           satisfy the computed comparison.
        """
        if valid == 1:
            if condition == 0:
                mask_valid_labels = attribute < rule
            else:
                mask_valid_labels = attribute > rule
        else:
            if condition == 0:
                mask_valid_labels = attribute > rule
            else:
                mask_valid_labels = attribute < rule
        return labels[mask_valid_labels]

    def get_filter_attributes(self, attributes, attribute, valid, rule, condition):
        """
        Filters rows of the attributes DataFrame using the same two-flag logic as
        get_filter_labels, then drops the split attribute column from the result.

        The effective row filter follows the same (valid, condition) truth table:

            valid=1, condition=0  →  keep rows where attribute < rule
            valid=1, condition=1  →  keep rows where attribute > rule
            valid=0, condition=0  →  keep rows where attribute > rule
            valid=0, condition=1  →  keep rows where attribute < rule

        After filtering, the column used as the split criterion is removed because
        it has already been consumed by the current decision node and should not be
        considered again in child nodes.

        Parameters:
            attributes (pandas.DataFrame): The full feature matrix at the current
                                           tree level.
            attribute (str): Column name used as the split criterion.
            valid (int): Branch selector. 1 for the primary direction defined by
                         `condition`; 0 for the complementary direction.
            rule (float): Threshold value for the row filter.
            condition (int): Comparison direction flag. 0 means primary is `<`;
                             1 means primary is `>`.

        Returns:
            pandas.DataFrame: Filtered subset of `attributes` with the `attribute`
                              column removed.
        """
        if valid == 1:
            if condition == 0:
                attributes = attributes[attributes[attribute] < rule]
            else:
                attributes = attributes[attributes[attribute] > rule]
        else:
            if condition == 0:
                attributes = attributes[attributes[attribute] > rule]
            else:
                attributes = attributes[attributes[attribute] < rule]
        return attributes.drop(attribute, axis=1)

    def get_gini_impurity(self, labels):
        """
        Calculates the Gini impurity for a set of class labels.

        Gini impurity measures the probability that a randomly chosen sample
        would be incorrectly classified if it were labelled according to the
        class distribution in `labels`. It is defined as:

            G = sum_k p_k * (1 - p_k)

        where p_k is the proportion of samples belonging to class k.

        Parameters:
            labels (numpy.ndarray): 1-D array of class labels for a node's sample set.

        Returns:
            float: Gini impurity in the range [0, 0.5] for binary problems, or
                   [0, (K-1)/K] in general. Returns 0 if `labels` is empty (pure
                   or empty node convention).
        """
        if len(labels) == 0:
            return 0
        counts_unique_labels = np.unique(labels, return_counts=True)[1]
        total_labels = sum(counts_unique_labels)
        gini_impurity = sum(
            (counts_unique_labels / total_labels)
            * (1 - (counts_unique_labels / total_labels))
        )
        return gini_impurity

    def get_most_repeated_value(self, labels):
        """
        Returns the majority class label in the array (mode).

        Used to assign a predicted class to leaf nodes where the Gini impurity
        is 0 or when only one attribute column remains and the primary branch
        (valid=1) is being resolved.

        Parameters:
            labels (numpy.ndarray): 1-D array of class labels.

        Returns:
            scalar: The label value with the highest frequency. If multiple labels
                    share the maximum count, the one that appears first in the sorted
                    unique array is returned (numpy.argmax tie-breaking behaviour).
        """
        labels_unique, labels_count = np.unique(labels, return_counts=True)
        return labels_unique[np.argmax(labels_count)]

    def get_less_repeated_value(self, labels):
        """
        Returns the minority class label in the array.

        Used exclusively in the single-attribute fallback path of generate_nodes,
        where the complementary branch (valid=0) is assigned the least frequent
        class to differentiate it from the primary branch prediction.

        Parameters:
            labels (numpy.ndarray): 1-D array of class labels.

        Returns:
            scalar: The label value with the lowest frequency. If multiple labels
                    share the minimum count, the one that appears first in the sorted
                    unique array is returned (numpy.argmin tie-breaking behaviour).
        """
        labels_unique, labels_count = np.unique(labels, return_counts=True)
        return labels_unique[np.argmin(labels_count)]

    def get_metrics_attribute(self, attribute, labels):
        """
        Finds the optimal binary split threshold for a single attribute by minimising
        Gini impurity over all candidate midpoints between consecutive sorted values.

        Candidate thresholds are the midpoints between every pair of adjacent distinct
        values in the sorted attribute array. For each candidate, both the primary
        branch (attribute > threshold, condition=1) and the complementary branch
        (attribute < threshold, condition=0) are evaluated. The winning branch for
        each threshold is the one with lower Gini impurity; ties are broken in favour
        of the side that covers more samples.

        A minimum-sample guard (`min_samples = 25% of total`) is enforced: a candidate
        split is only accepted if its matching sample count exceeds the previous best
        OR is at least `min_samples`. This prevents heavily imbalanced splits.

        Parameters:
            attribute (numpy.ndarray): 1-D array of attribute values (unsorted on input;
                                       sorted descending internally).
            labels (numpy.ndarray): 1-D array of class labels in the same original order
                                    as `attribute`.

        Returns:
            tuple:
                rule_attribute (list): Two-element list [threshold, condition] where
                                       `threshold` is the optimal split value and
                                       `condition` is 0 (primary side uses `<`) or
                                       1 (primary side uses `>`).
                gini_impurity (float): Gini impurity of the winning branch at the
                                       optimal threshold. Starts at 1.0 and is
                                       progressively minimised.
                old_matching_samples (int): Number of samples in the winning branch
                                            at the optimal threshold.
        """
        sorted_indices = np.argsort(attribute)[::-1]
        attribute_sorted = attribute[sorted_indices]
        labels_sorted = labels[sorted_indices]
        old_average = 0
        rule_attribute = []
        gini_impurity = 1
        old_matching_samples = 0
        min_samples = int((len(labels) * 25) / 100)

        for i in range(len(attribute_sorted) - 1):
            average = (attribute_sorted[i] + attribute_sorted[i + 1]) / 2
            if average != old_average:
                old_average = average
                labels_satisfy_less_operation = self.get_filter_labels(
                    attribute_sorted, labels_sorted, 1, average, 0
                )
                labels_satisfy_great_operation = self.get_filter_labels(
                    attribute_sorted, labels_sorted, 1, average, 1
                )
                gini_impurity_great_operation = self.get_gini_impurity(
                    labels_satisfy_great_operation
                )
                gini_impurity_less_operation = self.get_gini_impurity(
                    labels_satisfy_less_operation
                )
                number_matching_great_sample = len(labels_satisfy_great_operation)
                number_matching_less_sample = len(labels_satisfy_less_operation)

                if (
                    gini_impurity_great_operation < gini_impurity_less_operation
                    and gini_impurity_great_operation <= gini_impurity
                    and (
                        number_matching_great_sample > old_matching_samples
                        or number_matching_great_sample >= min_samples
                    )
                ):
                    gini_impurity = gini_impurity_great_operation
                    rule_attribute = [average, 1]
                    old_matching_samples = number_matching_great_sample

                elif (
                    gini_impurity_less_operation < gini_impurity_great_operation
                    and gini_impurity_less_operation <= gini_impurity
                    and (
                        number_matching_less_sample > old_matching_samples
                        or number_matching_less_sample >= min_samples
                    )
                ):
                    gini_impurity = gini_impurity_less_operation
                    rule_attribute = [average, 0]
                    old_matching_samples = number_matching_less_sample

                elif (
                    gini_impurity_great_operation == gini_impurity_less_operation
                    and number_matching_less_sample > number_matching_great_sample
                    and gini_impurity_less_operation <= gini_impurity
                    and (
                        number_matching_less_sample > old_matching_samples
                        or number_matching_less_sample >= min_samples
                    )
                ):
                    gini_impurity = gini_impurity_less_operation
                    rule_attribute = [average, 0]
                    old_matching_samples = number_matching_less_sample

                elif (
                    gini_impurity_great_operation == gini_impurity_less_operation
                    and number_matching_great_sample > number_matching_less_sample
                    and gini_impurity_great_operation <= gini_impurity
                    and (
                        number_matching_great_sample > old_matching_samples
                        or number_matching_great_sample >= min_samples
                    )
                ):
                    gini_impurity = gini_impurity_great_operation
                    rule_attribute = [average, 1]
                    old_matching_samples = number_matching_great_sample

        return rule_attribute, gini_impurity, old_matching_samples

    def get_properties_best_attribute(self, attributes, labels):
        """
        Selects the best feature and its optimal split rule from the current attribute
        set by comparing Gini impurity across all columns.

        For each column in `attributes`, get_metrics_attribute is called to obtain
        the best threshold and its impurity. The column that achieves the lowest
        impurity is selected. Ties are broken by preferring the attribute whose
        winning branch covers more samples, subject to the same 25%-minimum guard
        used inside get_metrics_attribute.

        Parameters:
            attributes (pandas.DataFrame): Feature matrix at the current tree level.
                                           Each column is a candidate split attribute.
            labels (numpy.ndarray): Class labels for the current sample subset.

        Returns:
            tuple:
                best_attribute (str or int): Column name of the selected feature.
                rule_best_attribute (list): Two-element list [threshold, condition]
                                            representing the optimal split for the
                                            selected feature (see get_metrics_attribute).
        """
        best_attribute = 0
        rule_best_attribute = []
        gini_impurity_best_attribute = 1
        great_matching_samples = 0
        min_samples = int((len(labels) * 25) / 100)

        for i in range(0, len(attributes.columns)):
            attribute = attributes.iloc[:, i].values
            rule_attribute, gini_impurity_attribute, matching_samples = (
                self.get_metrics_attribute(attribute, labels)
            )
            print(rule_attribute)
            print(gini_impurity_attribute)
            print(matching_samples)
            if gini_impurity_attribute <= gini_impurity_best_attribute and (
                matching_samples > great_matching_samples
                or great_matching_samples >= min_samples
            ):
                gini_impurity_best_attribute = gini_impurity_attribute
                rule_best_attribute = rule_attribute
                great_matching_samples = matching_samples
                best_attribute = attributes.columns[i]

        return best_attribute, rule_best_attribute

    def fit(self, attributes, labels):
        """
        Trains the decision tree on the provided dataset.

        Finds the best root-level split, constructs the root node, appends it to
        self.tree, and then recursively builds all descendant nodes via
        generate_nodes.

        Parameters:
            attributes (pandas.DataFrame): Training feature matrix where each column
                                           is an attribute and each row is a sample.
            labels (numpy.ndarray): 1-D array of class labels for each training sample.

        Returns:
            list: The fully constructed tree as a list of node dictionaries in the
                  order they were appended (breadth-first traversal order). Each
                  element is a dict produced by get_node().
        """
        root_attribute, rule = self.get_properties_best_attribute(attributes, labels)
        root = self.get_node("root", -1, 0, root_attribute, rule[0], rule[1])
        self.tree.append(root)
        self.generate_nodes(root, attributes, labels)
        return self.tree

    def generate_nodes(self, last_node, attributes, labels):
        """
        Recursively generates and appends child nodes for a given parent decision node.

        Two branches are created from each internal node, one for each value of `valid`
        (0 and 1). The parent's split attribute and rule ([threshold, condition]) are
        read from last_node["value"], and get_filter_labels / get_filter_attributes are
        used to route samples into each branch.

        Branch resolution logic:
        - If more than one attribute column remains:
            * If the filtered labels have Gini impurity == 0, create a leaf
              ("conclusion") node whose predicted label is the majority class.
            * Otherwise, find the best attribute for the child subset and create a
              new internal ("decision") node, then recurse.
        - If only one attribute column remains (base case):
            * valid=1 branch → leaf node predicting the majority class.
            * valid=0 branch → leaf node predicting the minority class (least
              repeated), to differentiate the two leaves when a single attribute
              can no longer produce a meaningful split.

        All new nodes are appended to self.tree as a side effect; this method does
        not return a value.

        Parameters:
            last_node (dict): Parent node dictionary. Must contain:
                              last_node["value"][0] — split attribute column name,
                              last_node["value"][1] — [threshold, condition] list.
            attributes (pandas.DataFrame): Feature matrix available at this depth.
            labels (numpy.ndarray): Class labels for the samples reaching this node.

        Returns:
            None
        """
        attribute = last_node["value"][0]
        rule = last_node["value"][1][0]
        condition = last_node["value"][1][1]
        print(rule, " : ", condition)
        if len(attributes.columns) > 1:
            for i in range(0, 2):
                labels_attribute = self.get_filter_labels(
                    attributes[attribute], labels, i, rule, condition
                )
                print(labels_attribute)
                new_attributes = self.get_filter_attributes(
                    attributes, attribute, i, rule, condition
                )
                gini_impurity = self.get_gini_impurity(labels_attribute)
                if gini_impurity == 0:
                    label = self.get_most_repeated_value(labels_attribute)
                    new_node = self.get_node("conclusion", attribute, i, 0, label, 0)
                    self.tree.append(new_node)
                else:
                    new_best_attribute, rule_best_attribute = (
                        self.get_properties_best_attribute(
                            new_attributes, labels_attribute
                        )
                    )
                    print(rule_best_attribute)
                    new_node = self.get_node(
                        "decision",
                        attribute,
                        i,
                        new_best_attribute,
                        rule_best_attribute[0],
                        rule_best_attribute[1],
                    )
                    self.tree.append(new_node)
                    self.generate_nodes(new_node, new_attributes, labels_attribute)
        else:
            labels_attribute = self.get_filter_labels(
                attributes[attribute], labels, 1, rule, condition
            )
            label = self.get_most_repeated_value(labels_attribute)
            new_node = self.get_node("conclusion", attribute, 1, 0, label, 0)
            self.tree.append(new_node)
            labels_attribute = self.get_filter_labels(
                attributes[attribute], labels, 0, rule, condition
            )
            label = self.get_less_repeated_value(labels_attribute)
            new_node = self.get_node("conclusion", attribute, 0, 0, label, 0)
            self.tree.append(new_node)

    def get_node(self, type, parent_attribute, filter, idx, rule, condition):
        """
        Constructs and returns a node dictionary representing a single element
        in the decision tree.

        Node structure:
            {
                "type":             str   — "root", "decision", or "conclusion",
                "parent_attribute": str or int — the attribute column name that was
                                    split to produce this node's branch (-1 for root),
                "filter":           int   — the branch direction taken from the parent:
                                    1 means the sample satisfied the primary comparison
                                    (valid=1 side); 0 means the complementary side,
                "value":            list  — [idx, [rule, condition]] where:
                                    • idx: column name to split on (decision/root nodes)
                                           or 0 (conclusion nodes, unused),
                                    • rule: numeric split threshold (decision/root) or
                                            predicted class label (conclusion nodes),
                                    • condition: comparison direction flag (0 = primary
                                                 side uses `<`, 1 = primary uses `>`;
                                                 always 0 for conclusion nodes).
            }

        Parameters:
            type (str): Node role — "root" for the initial split node, "decision" for
                        internal split nodes, "conclusion" for leaf/prediction nodes.
            parent_attribute (str or int): Column name of the attribute split that led
                                           to this node, or -1 for the root node.
            filter (int): Branch direction from the parent: 1 for the primary branch
                          (valid=1), 0 for the complementary branch (valid=0).
            idx (str or int): For "root" and "decision" nodes, the column name of the
                              attribute to split on. For "conclusion" nodes, pass 0
                              (the predicted label is stored in `rule` instead).
            rule (float or scalar): For "root" and "decision" nodes, the numeric split
                                    threshold. For "conclusion" nodes, the predicted
                                    class label.
            condition (int): Comparison direction flag used together with `valid` in
                             get_filter_labels and get_filter_attributes. 0 means the
                             primary (valid=1) side applies `<`; 1 means it applies `>`.
                             Pass 0 for "conclusion" nodes (unused).

        Returns:
            dict: A node dictionary with keys "type", "parent_attribute", "filter",
                  and "value" (a list of [idx, [rule, condition]]).
        """
        return {
            "type": type,
            "parent_attribute": parent_attribute,
            "filter": filter,
            "value": [idx, [rule, condition]],
        }

**CHARGE DATASET**

In [61]:
!wget https://raw.githubusercontent.com/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/main/Sources/iris.csv
df=pd.read_csv('iris.csv')
sheet = sheets.InteractiveSheet(df=df)

--2026-06-25 23:23:20--  https://raw.githubusercontent.com/LuisFelipeVelasco/ML_And_Generative_AI_Experiments/main/Sources/iris.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3858 (3.8K) [text/plain]
Saving to: ‘iris.csv.4’

iris.csv.4          100%[===================>]   3.77K  --.-KB/s    in 0s      

2026-06-25 23:23:20 (47.1 MB/s) - ‘iris.csv.4’ saved [3858/3858]

https://docs.google.com/spreadsheets/d/1WI-nuwtXAyETqLycf6mF6oq0TRnlFk-l4IASh8e47vQ/edit#gid=0


**TEST CLASS**

In [112]:
labels=df["species"].values
attributes=df.iloc[:,:-1]
model=BinaryDecisionTree()
tree=model.fit(attributes,labels)
print(tree)

[np.float64(4.9), 0]
0.0
16
[np.float64(3.8499999999999996), 1]
0.0
6
[np.float64(1.55), 0]
0.0
37
[np.float64(0.35), 0]
0.0
41
0.35  :  0
['setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa' 'setosa'
 'setosa' 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'versicolor' 'versicolor' 'versicolor' 'versicolor' 'versicolor'
 'virginica' 'virginica' 'virginica' 'virginica' 'virginica' 'virginica'
 'virginica' 'virginica' 'virginica' 'virginic

In [113]:
labels=df.iloc[:,4]
new_attributes=df.iloc[:,:4]
model=BinaryDecisionTree()
best_attribute,best_rule=model.get_properties_best_attribute(new_attributes,labels)
print(best_attribute)
print(best_rule)

[np.float64(4.9), 0]
0.0
16
[np.float64(3.8499999999999996), 1]
0.0
6
[np.float64(1.55), 0]
0.0
37
[np.float64(0.35), 0]
0.0
41
petal_width
[np.float64(0.35), 0]
